# Activity: Real-time Crypto Currency Price Quotes with WebSockets
Fill me in. 

> __Learning Objective:__ 
> 
> By the end of this activity, you will be able to:
> Three key learning objectives for this activity go here. 

Let's go!
___

## Setup, Data, and Prerequisites
First, we set up the computational environment by including the `Include.jl` file and loading any needed resources.

> The [`include(...)` command](https://docs.julialang.org/en/v1/base/base/#include) evaluates the contents of the input source file, `Include.jl`, in the notebook's global scope. The `Include.jl` file sets paths, loads required external packages, etc. For additional information on functions and types used in this material, see the [Julia programming language documentation](https://docs.julialang.org/en/v1/). 

Let's set up our code environment:

In [ ]:
include(joinpath(@__DIR__, "Include.jl")); # include the Include.jl file

### Data
For this activity, we will be using real-time cryptocurrency price data from the [Kraken exchange via their public WebSocket API](https://docs.kraken.com/api/docs/guides/spot-ws-intro). Let's start by specifcying the currency pair we want to monitor, the number of messages to receive, and the WebSocket URL.

In [ ]:
url = "wss://ws.kraken.com/v2"; # WebSocket URL for Kraken's public API
pair = "BTC/USD"; # Currency pair to monitor (Bitcoin to US Dollar)
number_of_messages = 100; # Number of messages to receive

___

## Let's implement our listener!
In this task, you will implement a WebSocket listener that connects to the Kraken WebSocket API and receives real-time price quotes for the specified cryptocurrency pair. 

Let's start by setting up the initial (subscribe) message to send to the WebSocket server to subscribe to the ticker feed for our chosen currency pair. We save this message in a variable called `subscription_message::Dict{String, String}`.

> __What's in a subscription message?__ A subscription message is a JSON-formatted message that tells the WebSocket server which data streams you want to receive. In this case, we want to subscribe to the `ticker` feed for the specified currency pair (e.g., BTC/USD).

In [ ]:
subscription_message = Dict(
    "method" => "subscribe",
    "params" => Dict("channel" => "ticker", "symbol" => [$(pair)]),
);

Now that we have our subscription message set up, we can proceed to implement the WebSocket listener to connect to the Kraken API and start receiving real-time price quotes.

> __What's going on in the code below?__ 
> 
> Explain the code that follows here. Make sure to describe the purpose of each section and how it contributes to the overall functionality of the WebSocket listener, link to relevant documentation as needed for external functions or types used.

So what do you think? Give it a try!

In [ ]:
WS.open(url) do ws
    WS.send(ws, JSON.json(subscription_message))
    seen = 0; # count of messages seen
    max_messages = number_of_messages; # number of messages to receive (then we close the connection)

    for raw in ws
        s = raw isa AbstractString ? raw : String(raw) # raw can be String or Vector{UInt8}; normalize to String

        # ok: parse the JSON message. If something goes wrong, log a warning and skip to the next message
        msg = try
            JSON.parse(s)
        catch err
            @warn "failed to parse websocket payload" error=err raw=s
            continue
        end

        # Kraken sends a status update on connect; just ignore or print it
        if msg isa JSON.Object{String, Any}
            channel = get(msg, "channel", nothing) # get the channel field if it exists
            msg_type = get(msg, "type", nothing) # get the type field if it exists

            # here: we can handle different message types as desired
            if channel == "status"
                @info "status" msg # echo the status message
                continue
            elseif channel == "heartbeat"
                @info "heartbeat" msg # echo the heartbeat message
                continue
            elseif channel == "ticker" && msg_type == "update"
                @info "ticker update" msg
                
                # Your logic goes here!
                # process ticker data here if desired; Kraken packs updates in msg["data"]
                continue
            end
        end
        @info "unhandled payload" msg

        seen += 1 # update message count
        if seen >= max_messages
            @info "max message count reached; closing websocket" count=seen
            WS.close(ws) # close the WebSocket connection
            break
        end
    end
end